# YOLOv8m Fine-Tuning for Pomegranate Orchard Detection

**Runtime:** GPU (T4 recommended)  
**Dataset:** 142 real pomegranate orchard images with pseudo-labels  
**Base model:** Existing `yolov8_pomegranate.pt` (Turkish-trained)  
**Goal:** Improve detection on orchard/tree pomegranates

## 1. Install Dependencies

In [ ]:
!pip install -q --upgrade numpy
!pip install -q ultralytics==8.4.50
import os
os.kill(os.getpid(), 9)  # Restart Colab runtime so new packages load

## 2. Clone Repository & Set Up Dataset

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Samer-Gassouma/Pomegranate-Orchard-Monitor.git"
REPO_DIR = Path("Pomegranate-Orchard-Monitor")

# Clone if not already present
if not REPO_DIR.exists():
    !git clone {REPO_URL}
else:
    print("Repo already cloned")

# Fix data.yaml path for Colab environment
DATASET_YAML = REPO_DIR / "datasets/pomegranate_orchard/data.yaml"
assert DATASET_YAML.exists(), f"Dataset not found at {DATASET_YAML}"

# Update the path in data.yaml to absolute Colab path
yaml_content = DATASET_YAML.read_text()
# Replace any path line with the absolute path
fixed_yaml = yaml_content.replace(
    "path: ../datasets/pomegranate_orchard",
    f"path: {REPO_DIR.absolute()}/datasets/pomegranate_orchard"
)
DATASET_YAML.write_text(fixed_yaml)
print(f"Dataset ready: {DATASET_YAML}")

## 3. Inspect Dataset

In [ ]:
import glob

train_imgs = sorted(glob.glob(str(REPO_DIR / "datasets/pomegranate_orchard/images/train/*.jpg")))
val_imgs = sorted(glob.glob(str(REPO_DIR / "datasets/pomegranate_orchard/images/val/*.jpg")))

print(f"Train images: {len(train_imgs)}")
print(f"Val images:   {len(val_imgs)}")

# Show sample labels
sample_label = REPO_DIR / "datasets/pomegranate_orchard/labels/train" / (Path(train_imgs[0]).stem + ".txt")
print(f"\nSample label file ({sample_label.name}):")
if sample_label.exists():
    print(sample_label.read_text()[:500])

## 4. Fine-Tune YOLOv8m

**Config:**
- Base: existing Turkish-trained model  
- Epochs: 50  
- Img size: 640  
- Batch: 16  
- Augmentation: default (Mosaic, MixUp, HSV, flip, scale)  
- Patience: 10 (early stopping)

In [ ]:
from ultralytics import YOLO

# Load existing model as starting point
model_path = str(REPO_DIR / "pomegranate_models/yolov8_pomegranate.pt")
model = YOLO(model_path)

# Fine-tune
results = model.train(
    data=str(DATASET_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    device=0,  # GPU
    amp=True,  # Mixed precision for T4
    project="runs/orchard_finetune",
    name="yolov8m_orchard",
)

## 5. Evaluate on Validation Set

In [ ]:
# Load best model from training
best_model = YOLO("runs/orchard_finetune/yolov8m_orchard/weights/best.pt")

# Validate
metrics = best_model.val(data=str(DATASET_YAML))

print(f"mAP@0.5:    {metrics.box.map50:.4f}")
print(f"mAP@0.5:95: {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

## 6. Test on Sample Images

In [ ]:
import random
from PIL import Image
from IPython.display import display

# Test on random validation images
test_samples = random.sample(val_imgs, min(5, len(val_imgs)))

for img_path in test_samples:
    results = best_model(img_path, conf=0.25, verbose=False)
    annotated = results[0].plot()
    display(Image.fromarray(annotated))
    print(f"{Path(img_path).name}: {len(results[0].boxes)} detections\n")

## 7. Export & Download Model

In [ ]:
# Export to TorchScript and ONNX
best_model.export(format="torchscript")
best_model.export(format="onnx")

# Show files
!ls -lh runs/orchard_finetune/yolov8m_orchard/weights/

# Download link
from google.colab import files
files.download("runs/orchard_finetune/yolov8m_orchard/weights/best.pt")

## 8. (Optional) Push to GitHub

Replace the old model in your repo with the new one, then commit and push.

In [ ]:
# Copy best model to repo and push (requires git auth)
!cp runs/orchard_finetune/yolov8m_orchard/weights/best.pt {REPO_DIR}/pomegranate_models/yolov8_pomegranate.pt

# Uncomment below after setting up git credentials
# !cd {REPO_DIR} && git add pomegranate_models/yolov8_pomegranate.pt && git commit -m "Fine-tuned YOLOv8m on orchard dataset" && git push